# Full Deepfake Audio Detection Model

This notebook contains the **complete implementation** of the Neuro-Fuzzy Deepfake Audio Detection Model.

## Architecture Overview

```
Input Spectrogram → CNN Encoder → NFIS → Classifier → Output
                    (features)   (refined + uncertainty)
```

### Components:
1. **CNN Encoder**: Extracts features from audio spectrograms
2. **Neuro-Fuzzy Inference System (NFIS)**: Enhances features and provides uncertainty estimation
3. **Classifier**: Makes the final deepfake/real classification
4. **DeepfakeDetector**: Complete end-to-end model

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Any, Tuple, List

## 1. CNN Encoder Module

In [ ]:
"""
CNN Encoder module for extracting features from audio spectrograms.
"""

import torch
import torch.nn as nn
from typing import Dict, Any


class ConvBlock(nn.Module):
    """
    A convolutional block consisting of Conv2D, ReLU, BatchNorm, and MaxPool.
    
    Attributes:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
    """
    
    def __init__(self, in_channels: int, out_channels: int) -> None:
        """
        Initialize the ConvBlock.
        
        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
        """
        super(ConvBlock, self).__init__()
        
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            padding=1,
        )
        self.relu = nn.ReLU()
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the convolutional block.
        
        Args:
            x: Input tensor of shape (batch, in_channels, height, width).
            
        Returns:
            Output tensor of shape (batch, out_channels, height/2, width/2).
        """
        x = self.conv(x)
        x = self.relu(x)
        x = self.batch_norm(x)
        x = self.max_pool(x)
        return x


class CNNEncoder(nn.Module):
    """
    CNN Encoder for extracting features from audio spectrograms.
    
    Consists of 3 convolutional blocks followed by adaptive average pooling.
    
    Attributes:
        input_channels (int): Number of input channels.
        feature_dim (int): Dimension of the output feature vector.
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """
        Initialize the CNNEncoder.
        
        Args:
            config: Configuration dictionary containing model parameters.
        """
        super(CNNEncoder, self).__init__()
        
        self.input_channels = config.get("input_channels", 1)
        self.feature_dim = config.get("feature_dim", 128)
        
        # Three convolutional blocks
        self.conv_block1 = ConvBlock(self.input_channels, 32)
        self.conv_block2 = ConvBlock(32, 64)
        self.conv_block3 = ConvBlock(64, 128)
        
        # Adaptive average pooling to get fixed-size output
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layer to project to feature_dim
        self.fc = nn.Linear(128, self.feature_dim)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the CNN encoder.
        
        Args:
            x: Input tensor of shape (batch, input_channels, 128, 128).
            
        Returns:
            Output tensor of shape (batch, feature_dim).
        """
        # Pass through convolutional blocks
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        
        # Adaptive average pooling
        x = self.adaptive_pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Project to feature dimension
        x = self.fc(x)
        
        return x


## 2. Neuro-Fuzzy Inference System (NFIS)

In [ ]:
"""
Advanced Neuro-Fuzzy Inference System (NFIS) for deepfake detection.

This module implements a differentiable Fuzzy Inference System (FIS) that:
- Enhances feature representation from CNN
- Learns fuzzy rules automatically
- Models uncertainty explicitly
- Provides interpretable outputs
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Any, Tuple


class NeuroFuzzyInferenceSystem(nn.Module):
    """
    Advanced Neuro-Fuzzy Inference System (NFIS) layer.
    
    This layer implements a complete fuzzy inference system with:
    1. Fuzzification Layer: Multiple Gaussian membership functions per feature
    2. Rule Base: Learnable fuzzy rules aggregating memberships
    3. Inference Engine: Rule strength computation with normalization
    4. Defuzzification: Mapping rule activations back to feature space
    5. Uncertainty Estimation: Entropy-based uncertainty scoring
    6. Interpretability: Rule explanation capabilities
    
    Attributes:
        feature_dim (int): Dimension of input features.
        num_memberships (int): Number of membership functions per feature.
        num_rules (int): Number of fuzzy rules in the rule base.
        centers (nn.Parameter): Learnable centers for Gaussian membership functions.
        sigmas (nn.Parameter): Learnable sigmas for Gaussian membership functions.
        rule_weights (nn.Parameter): Learnable weights for rule aggregation.
        rule_to_feature (nn.Parameter): Learnable mapping from rules to feature space.
    """
    
    def __init__(
        self,
        feature_dim: int = 128,
        num_memberships: int = 3,
        num_rules: int = 32,
        sigma_init: float = 0.5,
    ) -> None:
        """
        Initialize the Neuro-Fuzzy Inference System.
        
        Args:
            feature_dim: Dimension of input features (default: 128).
            num_memberships: Number of Gaussian membership functions per feature (default: 3).
            num_rules: Number of fuzzy rules in the rule base (default: 32).
            sigma_init: Initial value for sigma parameters (default: 0.5).
        """
        super(NeuroFuzzyInferenceSystem, self).__init__()
        
        self.feature_dim = feature_dim
        self.num_memberships = num_memberships
        self.num_rules = num_rules
        self.sigma_init = sigma_init
        
        # =====================================================================
        # 1. FUZZIFICATION LAYER PARAMETERS
        # Multiple Gaussian membership functions per feature
        # =====================================================================
        # centers: (feature_dim, num_memberships)
        self.centers = nn.Parameter(torch.Tensor(feature_dim, num_memberships))
        # sigmas: (feature_dim, num_memberships)
        self.sigmas = nn.Parameter(torch.Tensor(feature_dim, num_memberships))
        
        # =====================================================================
        # 2. RULE BASE PARAMETERS
        # Learnable combinations of feature memberships
        # =====================================================================
        # rule_weights: (num_rules, feature_dim, num_memberships)
        self.rule_weights = nn.Parameter(torch.Tensor(num_rules, feature_dim, num_memberships))
        
        # =====================================================================
        # 4. DEFUZZIFICATION PARAMETERS
        # Map rule activations back to feature space
        # =====================================================================
        # rule_to_feature: (num_rules, feature_dim)
        self.rule_to_feature = nn.Parameter(torch.Tensor(num_rules, feature_dim))
        
        # Initialize all parameters
        self._initialize_parameters()
        
        # Small epsilon for numerical stability
        self.eps = 1e-8
    
    def _initialize_parameters(self) -> None:
        """
        Initialize all learnable parameters with appropriate distributions.
        
        - Centers: Uniformly distributed based on expected input range
        - Sigmas: Initialized to sigma_init with small variation
        - Rule weights: Xavier initialization
        - Rule-to-feature mapping: Xavier initialization
        """
        # Initialize centers uniformly across expected feature range
        nn.init.uniform_(self.centers, -1.0, 1.0)
        
        # Initialize sigmas with positive values
        nn.init.constant_(self.sigmas, self.sigma_init)
        # Add small variation
        self.sigmas.data += torch.randn_like(self.sigmas.data) * 0.1
        # Ensure sigmas stay positive
        self.sigmas.data = torch.clamp(self.sigmas.data, min=0.1)
        
        # Initialize rule weights using Xavier initialization
        nn.init.xavier_uniform_(self.rule_weights)
        
        # Initialize rule-to-feature mapping using Xavier initialization
        nn.init.xavier_uniform_(self.rule_to_feature)
    
    def _fuzzification(self, x: torch.Tensor) -> torch.Tensor:
        """
        Fuzzification Layer: Compute membership degrees for each feature.
        
        Implements multiple Gaussian membership functions per feature:
        mu = exp(- (x - centers)^2 / (2 * sigma^2))
        
        Args:
            x: Input tensor of shape (batch_size, feature_dim).
            
        Returns:
            Membership degrees tensor of shape (batch_size, feature_dim, num_memberships).
        """
        batch_size = x.shape[0]
        
        # Reshape x for broadcasting: (batch_size, feature_dim, 1)
        x_expanded = x.unsqueeze(-1)
        
        # Compute squared differences: (batch_size, feature_dim, num_memberships)
        diff_squared = (x_expanded - self.centers.unsqueeze(0)) ** 2
        
        # Compute denominator with numerical stability: (feature_dim, num_memberships)
        sigma_squared = self.sigmas ** 2 + self.eps
        denominator = 2.0 * sigma_squared
        
        # Compute Gaussian membership: (batch_size, feature_dim, num_memberships)
        exponent = -diff_squared / denominator
        memberships = torch.exp(exponent)
        
        return memberships
    
    def _rule_aggregation(self, memberships: torch.Tensor) -> torch.Tensor:
        """
        Rule Base: Aggregate memberships using learnable rule weights.
        
        Implements fuzzy rules as learnable combinations of feature memberships.
        Uses product operation (differentiable AND) with log-space computation
        for numerical stability.
        
        Args:
            memberships: Membership degrees of shape (batch_size, feature_dim, num_memberships).
            
        Returns:
            Rule activations of shape (batch_size, num_rules).
        """
        batch_size = memberships.shape[0]
        
        # Apply softmax to rule_weights to ensure they represent valid importance weights
        # Shape: (num_rules, feature_dim, num_memberships)
        rule_weights_normalized = F.softmax(self.rule_weights, dim=-1)
        
        # Weighted membership: (batch_size, num_rules, feature_dim)
        # Multiply memberships by rule weights and sum over memberships
        weighted_memberships = memberships.unsqueeze(1) * rule_weights_normalized.unsqueeze(0)
        # Sum over membership dimension: (batch_size, num_rules, feature_dim)
        weighted_sum = weighted_memberships.sum(dim=-1)
        
        # Use log-sum-exp trick for numerical stability in product operation
        # Product of memberships = exp(sum(log(memberships)))
        # Add epsilon to avoid log(0)
        log_memberships = torch.log(weighted_sum + self.eps)
        
        # Sum log-memberships across features (product in original space)
        # Shape: (batch_size, num_rules)
        log_rule_strengths = log_memberships.sum(dim=-1)
        
        # Convert back to linear space
        rule_activations = torch.exp(log_rule_strengths)
        
        return rule_activations
    
    def _inference_engine(self, rule_activations: torch.Tensor) -> torch.Tensor:
        """
        Inference Engine: Normalize rule activations using softmax.
        
        Ensures numerical stability through log-space computations.
        
        Args:
            rule_activations: Raw rule activations of shape (batch_size, num_rules).
            
        Returns:
            Normalized rule activations of shape (batch_size, num_rules).
        """
        # Apply softmax for normalization with numerical stability
        normalized_rules = F.softmax(rule_activations, dim=-1)
        
        return normalized_rules
    
    def _defuzzification(self, normalized_rules: torch.Tensor) -> torch.Tensor:
        """
        Defuzzification / Feature Projection: Map rule activations to feature space.
        
        Computes: refined_features = rule_activations @ rule_to_feature
        
        Args:
            normalized_rules: Normalized rule activations of shape (batch_size, num_rules).
            
        Returns:
            Refined features of shape (batch_size, feature_dim).
        """
        # Matrix multiplication: (batch_size, num_rules) @ (num_rules, feature_dim)
        # Result: (batch_size, feature_dim)
        refined_features = torch.matmul(normalized_rules, self.rule_to_feature)
        
        return refined_features
    
    def _compute_uncertainty(self, normalized_rules: torch.Tensor) -> torch.Tensor:
        """
        Uncertainty Estimation: Compute uncertainty score using entropy.
        
        Higher entropy indicates more uniform rule activation (higher uncertainty).
        Lower entropy indicates confident rule activation (lower uncertainty).
        
        Args:
            normalized_rules: Normalized rule activations of shape (batch_size, num_rules).
            
        Returns:
            Uncertainty scores of shape (batch_size, 1).
        """
        # Compute entropy: H = -sum(p * log(p))
        # Add epsilon to avoid log(0)
        log_probs = torch.log(normalized_rules + self.eps)
        entropy = -torch.sum(normalized_rules * log_probs, dim=-1, keepdim=True)
        
        # Normalize entropy by maximum possible entropy (log(num_rules))
        max_entropy = torch.log(torch.tensor(float(self.num_rules), device=entropy.device) + self.eps)
        normalized_uncertainty = entropy / (max_entropy + self.eps)
        
        return normalized_uncertainty
    
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass through the Neuro-Fuzzy Inference System.
        
        Args:
            x: Input tensor of shape (batch_size, feature_dim).
            
        Returns:
            Dictionary containing:
                - refined_features: Tensor of shape (batch_size, feature_dim)
                - uncertainty: Tensor of shape (batch_size, 1)
                - rule_activations: Tensor of shape (batch_size, num_rules)
        """
        # 1. Fuzzification: Compute membership degrees
        memberships = self._fuzzification(x)  # (batch_size, feature_dim, num_memberships)
        
        # 2. Rule Aggregation: Compute rule strengths
        rule_activations_raw = self._rule_aggregation(memberships)  # (batch_size, num_rules)
        
        # 3. Inference: Normalize rule activations
        normalized_rules = self._inference_engine(rule_activations_raw)  # (batch_size, num_rules)
        
        # 4. Defuzzification: Map to feature space
        refined_features = self._defuzzification(normalized_rules)  # (batch_size, feature_dim)
        
        # 5. Uncertainty Estimation
        uncertainty = self._compute_uncertainty(normalized_rules)  # (batch_size, 1)
        
        return {
            "refined_features": refined_features,
            "uncertainty": uncertainty,
            "rule_activations": normalized_rules,
        }
    
    def explain_rules(self, top_k: int = 5) -> Dict[str, Any]:
        """
        Generate interpretable descriptions of fuzzy rules.
        
        Analyzes the learned rule weights to identify:
        - Top contributing features per rule
        - Dominant membership functions for each feature-rule pair
        
        Args:
            top_k: Number of top features to report per rule (default: 5).
            
        Returns:
            Dictionary containing interpretable rule descriptions:
                - rules: List of dictionaries with rule details
                - feature_importance: Global feature importance across rules
        """
        # Get rule weights: (num_rules, feature_dim, num_memberships)
        rule_weights = self.rule_weights.detach().cpu()
        
        # Compute feature importance per rule (sum over memberships)
        feature_importance_per_rule = rule_weights.abs().sum(dim=-1)  # (num_rules, feature_dim)
        
        # Get dominant membership per feature-rule pair
        dominant_memberships = rule_weights.argmax(dim=-1)  # (num_rules, feature_dim)
        
        rules_description = []
        
        for rule_idx in range(self.num_rules):
            # Get top-k features for this rule
            rule_importance = feature_importance_per_rule[rule_idx]
            top_features = torch.topk(rule_importance, k=min(top_k, self.feature_dim))
            
            feature_contributions = []
            for feat_idx, importance in zip(top_features.indices.tolist(), top_features.values.tolist()):
                dom_membership = dominant_memberships[rule_idx, feat_idx].item()
                center_val = self.centers[feat_idx, dom_membership].item()
                sigma_val = self.sigmas[feat_idx, dom_membership].item()
                
                feature_contributions.append({
                    "feature_index": feat_idx,
                    "importance": float(importance),
                    "dominant_membership": dom_membership,
                    "center": float(center_val),
                    "sigma": float(sigma_val),
                })
            
            rules_description.append({
                "rule_index": rule_idx,
                "top_features": feature_contributions,
            })
        
        # Compute global feature importance (average across rules)
        global_feature_importance = feature_importance_per_rule.mean(dim=0)
        top_global_features = torch.topk(global_feature_importance, k=min(top_k, self.feature_dim))
        
        global_top_features = []
        for feat_idx, importance in zip(top_global_features.indices.tolist(), top_global_features.values.tolist()):
            global_top_features.append({
                "feature_index": feat_idx,
                "avg_importance": float(importance),
            })
        
        return {
            "rules": rules_description,
            "feature_importance": global_top_features,
            "num_rules": self.num_rules,
            "num_memberships": self.num_memberships,
            "feature_dim": self.feature_dim,
        }
    
    def get_membership_functions(self) -> Dict[str, torch.Tensor]:
        """
        Get the current membership function parameters.
        
        Returns:
            Dictionary containing:
                - centers: Tensor of shape (feature_dim, num_memberships)
                - sigmas: Tensor of shape (feature_dim, num_memberships)
        """
        return {
            "centers": self.centers.detach().cpu(),
            "sigmas": self.sigmas.detach().cpu(),
        }


# Backward compatibility alias
FuzzyLayer = NeuroFuzzyInferenceSystem


## 3. Classifier Module

In [ ]:
"""
Classifier module for the deepfake detection model.
"""

import torch
import torch.nn as nn
from typing import Dict, Any


class Classifier(nn.Module):
    """
    Fully connected classifier for deepfake detection.
    
    Architecture:
        128 -> 64 -> 1
    With ReLU activation and Sigmoid output.
    
    Attributes:
        feature_dim (int): Dimension of input features.
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """
        Initialize the Classifier.
        
        Args:
            config: Configuration dictionary containing model parameters.
        """
        super(Classifier, self).__init__()
        
        self.feature_dim = config.get("feature_dim", 128)
        
        # First fully connected layer: 128 -> 64
        self.fc1 = nn.Linear(self.feature_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        
        # Second fully connected layer: 64 -> 1
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the classifier.
        
        Args:
            x: Input tensor of shape (batch, feature_dim).
            
        Returns:
            Output tensor of shape (batch, 1) with classification probabilities.
        """
        # First layer
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        # Second layer
        x = self.fc2(x)
        x = self.sigmoid(x)
        
        return x.squeeze(-1)


## 4. Full Model: DeepfakeDetector

In [ ]:
"""
Full Model module combining CNN Encoder, Neuro-Fuzzy Inference System, and Classifier.
"""

import torch
import torch.nn as nn
from typing import Dict, Any, Tuple

from src.models.cnn_encoder import CNNEncoder
from src.fuzzy.fuzzy_layer import NeuroFuzzyInferenceSystem
from src.models.classifier import Classifier


class DeepfakeDetector(nn.Module):
    """
    Complete Deepfake Detection Model with Neuro-Fuzzy Inference System.
    
    Architecture:
        CNNEncoder -> NeuroFuzzyInferenceSystem -> Classifier
    
    The model returns a dictionary containing:
        - prediction: Classification probability
        - uncertainty: Uncertainty estimate from fuzzy layer
    
    Attributes:
        cnn_encoder (CNNEncoder): CNN feature extractor.
        fuzzy_layer (NeuroFuzzyInferenceSystem): Advanced fuzzy inference system.
        classifier (Classifier): Final classification layer.
    """
    
    def __init__(self, config: Dict[str, Any]) -> None:
        """
        Initialize the DeepfakeDetector.
        
        Args:
            config: Configuration dictionary containing all model parameters.
        """
        super(DeepfakeDetector, self).__init__()
        
        self.config = config
        
        # Get configuration values
        model_config = config.get("model", {})
        fuzzy_config = config.get("fuzzy", {})
        
        feature_dim = model_config.get("feature_dim", 128)
        num_memberships = fuzzy_config.get("num_memberships", 3)
        num_rules = fuzzy_config.get("num_rules", 32)
        sigma_init = fuzzy_config.get("sigma_init", 0.5)
        
        # CNN Encoder: (batch, 1, 128, 128) -> (batch, feature_dim)
        self.cnn_encoder = CNNEncoder(config)
        
        # Neuro-Fuzzy Inference System
        self.fuzzy_layer = NeuroFuzzyInferenceSystem(
            feature_dim=feature_dim,
            num_memberships=num_memberships,
            num_rules=num_rules,
            sigma_init=sigma_init,
        )
        
        # Classifier: (batch, feature_dim) -> (batch,)
        self.classifier = Classifier(config)
        
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass through the complete model.
        
        Args:
            x: Input tensor of shape (batch, input_channels, 128, 128).
            
        Returns:
            Dictionary containing:
                - prediction: Tensor of shape (batch,) with classification probabilities
                - uncertainty: Tensor of shape (batch_size, 1) with uncertainty estimates
        """
        # Extract features using CNN
        features = self.cnn_encoder(x)  # (batch, feature_dim)
        
        # Apply neuro-fuzzy inference system
        fuzzy_out = self.fuzzy_layer(features)
        
        # Get refined features and uncertainty
        refined_features = fuzzy_out["refined_features"]  # (batch, feature_dim)
        uncertainty = fuzzy_out["uncertainty"]  # (batch, 1)
        
        # Classify using refined features
        prediction = self.classifier(refined_features)  # (batch,)
        
        return {
            "prediction": prediction,
            "uncertainty": uncertainty,
        }
    
    def get_feature_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        """
        Get feature embeddings from the CNN encoder.
        
        Args:
            x: Input tensor of shape (batch, input_channels, 128, 128).
            
        Returns:
            Feature embeddings of shape (batch, feature_dim).
        """
        return self.cnn_encoder(x)
    
    def get_fuzzy_output(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Get full fuzzy layer output including refined features, uncertainty, and rule activations.
        
        Args:
            x: Input tensor of shape (batch, input_channels, 128, 128).
            
        Returns:
            Dictionary containing:
                - refined_features: Tensor of shape (batch, feature_dim)
                - uncertainty: Tensor of shape (batch, 1)
                - rule_activations: Tensor of shape (batch, num_rules)
        """
        features = self.cnn_encoder(x)
        return self.fuzzy_layer(features)
    
    def explain_rules(self) -> Dict[str, Any]:
        """
        Get interpretable explanations of the learned fuzzy rules.
        
        Returns:
            Dictionary containing rule explanations from the fuzzy layer.
        """
        return self.fuzzy_layer.explain_rules()


def build_model(config: Dict[str, Any]) -> DeepfakeDetector:
    """
    Build the deepfake detection model from configuration.
    
    Args:
        config: Configuration dictionary.
        
    Returns:
        Initialized DeepfakeDetector model.
    """
    return DeepfakeDetector(config)


def count_parameters(model: nn.Module) -> int:
    """
    Count the number of trainable parameters in a model.
    
    Args:
        model: PyTorch model.
        
    Returns:
        Number of trainable parameters.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Example Usage and Testing

In [ ]:
# Define configuration
config = {
    "input_channels": 1,
    "model": {"feature_dim": 128},
    "fuzzy": {"num_memberships": 3, "num_rules": 32, "sigma_init": 0.5}
}

# Build the model
model = build_model(config)
print(f"Model created! Parameters: {count_parameters(model):,}")

In [ ]:
# Test forward pass
sample_input = torch.randn(4, 1, 128, 128)
model.eval()
with torch.no_grad():
    output = model(sample_input)
print(f"Prediction shape: {output['prediction'].shape}")
print(f"Uncertainty shape: {output['uncertainty'].shape}")

## Summary

This notebook contains everything about the model:
- **Interpretability**: Fuzzy rules can be explained
- **Uncertainty Estimation**: Built-in via entropy
- **End-to-End Training**: All components differentiable